# Prototype VQLS with QuBlock and Qiskit

This tutorial solves a small linear system `A x = b` with a variational Qiskit ansatz while QuBlock evaluates the block-encoded projected map `A / alpha` and tracks quantum constraints.

This is an idealized, noiseless VQLS prototype. It avoids explicit block-encoding ancilla simulation. It does not implement Hadamard-test cost estimation, shot noise, controlled block encodings, or a hardware-ready VQLS circuit.

## Setup

Install tutorial dependencies with:

```bash
pip install -e .[tutorial]
```

The notebook also locates this repository's `src/` directory automatically.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "blockflow").exists() and (repo_root.parent / "src" / "blockflow").exists():
    repo_root = repo_root.parent
if (repo_root / "src" / "blockflow").exists():
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from qiskit import QuantumCircuit, qasm3
from qiskit.quantum_info import Statevector

from blockflow import (
    ApplyBlockEncodingQuantumStep,
    BlockEncoding,
    Capabilities,
    NumpyMatrixOperator,
    Program,
    ResourceEstimate,
    SemanticExecutor,
    StateVector,
)

np.set_printoptions(precision=6, suppress=True)

## Define a block-encoded linear system

We choose `A = 0.6 I + 0.3 X + 0.1 Z`. Its Pauli-LCU normalization is `alpha = 1.0`, which bounds its spectral norm and permits QuBlock synthesis.

In [ ]:
identity = np.eye(2, dtype=complex)
pauli_x = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
pauli_z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)

A = 0.6 * identity + 0.3 * pauli_x + 0.1 * pauli_z
alpha = 0.6 + 0.3 + 0.1
b = np.array([1.0, 0.0], dtype=complex)

block_encoding = BlockEncoding(
    op=NumpyMatrixOperator(A),
    alpha=alpha,
    resources=ResourceEstimate(
        ancilla_qubits_clean=2,
        depth=12,
        oracle_queries=1,
        postselections=1,
    ),
    capabilities=Capabilities(supports_adjoint=True, supports_circuit_recipe=True),
    synthesis_strategy="prep_select",
)

status = block_encoding.validate_quantum_constraints(require_verified=True)
print("A =\n", A)
print("alpha =", alpha)
print("spectral norm =", status.operator_norm)
print("constraint =", status.reason)

## Qiskit variational ansatz

The one-parameter ansatz `RY(theta)|0>` spans every real normalized one-qubit state, including this problem's exact normalized solution.

In [ ]:
def ansatz_state(theta: float) -> np.ndarray:
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    return Statevector.from_instruction(circuit).data


def evaluate_projected_block(theta: float):
    state = StateVector(ansatz_state(theta))
    program = Program([
        ApplyBlockEncodingQuantumStep(
            block_encoding,
            postselect=False,
            require_verified=True,
        )
    ])
    return SemanticExecutor().run(program, state)


def vqls_global_cost(theta: float) -> float:
    projected, _ = evaluate_projected_block(theta)
    y = projected.data
    denominator = float(np.vdot(y, y).real)
    overlap = abs(np.vdot(b, y)) ** 2
    return float(1.0 - overlap / denominator)


print("Initial cost =", vqls_global_cost(0.0))

The global VQLS cost used here is

`C(theta) = 1 - |<b|A|x(theta)>|^2 / <x(theta)|A†A|x(theta)>`.

Its minimum is zero when `A|x(theta)>` points along `|b>`. Scaling by `alpha` cancels from this cost, but QuBlock still applies and tracks it.

In [ ]:
result = minimize_scalar(
    vqls_global_cost,
    bounds=(-2.0 * np.pi, 2.0 * np.pi),
    method="bounded",
    options={"xatol": 1e-12},
)

theta_star = float(result.x)
x_vqls = ansatz_state(theta_star)
x_exact = np.linalg.solve(A, b)
x_exact = x_exact / np.linalg.norm(x_exact)

phase = np.vdot(x_exact, x_vqls)
x_vqls_aligned = x_vqls * np.exp(-1j * np.angle(phase))
residual = np.linalg.norm(A @ x_vqls - b * np.vdot(b, A @ x_vqls))

print("optimizer success =", result.success)
print("theta* =", theta_star)
print("final cost =", result.fun)
print("VQLS state =", x_vqls_aligned)
print("exact normalized solution =", x_exact)
print("state fidelity =", abs(np.vdot(x_exact, x_vqls)) ** 2)
print("directional residual =", residual)

## Inspect quantum constraints at the optimum

Now retain the successful projected branch. QuBlock normalizes that branch and reports its state-dependent postselection probability plus declared quantum resources.

In [ ]:
postselected_program = Program([
    ApplyBlockEncodingQuantumStep(
        block_encoding,
        postselect=True,
        require_verified=True,
    )
])
projected, report = SemanticExecutor().run(
    postselected_program,
    StateVector(x_vqls),
)

print("postselected A|x>/alpha =", projected.data)
print("target |b> =", b)
print("success probability =", report.cumulative_success_prob)
print("expected independent trials =", report.expected_trials)
print("trials for 99% confidence =", report.trials_for_confidence(0.99))
print("ancilla peak =", report.ancilla_clean_peak)
print("declared depth =", report.resources.depth)
print("oracle queries =", report.resources.oracle_queries)
print("constraints verified =", report.constraints_verified)

## Inspect synthesized block-encoding circuit

Synthesis is separate from semantic VQLS optimization. This small example can synthesize the Pauli-LCU block encoding and validate its OpenQASM 3 output with Qiskit.

In [ ]:
compiled = block_encoding.build_circuit(optimize=False)
qasm = block_encoding.export_openqasm(flavor="qasm3", optimize=False)
parsed = qasm3.loads(qasm)

print("compiled qubits =", compiled.num_qubits)
print("compiled gates =", len(compiled.gates))
print("Qiskit parsed qubits =", parsed.num_qubits)
print(qasm[:500])

## Plot the cost landscape

In [ ]:
angles = np.linspace(-2.0 * np.pi, 2.0 * np.pi, 300)
costs = np.array([vqls_global_cost(theta) for theta in angles])

plt.figure(figsize=(8, 4))
plt.plot(angles, costs, label="VQLS global cost")
plt.axvline(theta_star, color="tab:red", linestyle="--", label="optimizer result")
plt.xlabel("theta")
plt.ylabel("cost")
plt.legend()
plt.tight_layout()
plt.show()

## Large matrix-free VQLS prototype beyond full-circuit statevector simulation

Now use 18 system qubits and a generic diagonal operator. A generic `n`-qubit diagonal has up to `2^n` nonzero Pauli-Z strings. We explicitly compute those Pauli coefficients and derive the ancilla count for QuBlock's `prep_select` LCU construction instead of inventing a resource estimate.

For this deterministic example all `2^18` diagonal Pauli terms are nonzero, requiring 18 index ancillas plus one phase ancilla: 19 clean ancillas. A dense statevector for the resulting 37-qubit construction would require about 2 TiB. QuBlock stores only the 18-qubit system state, about 4 MiB, and applies `A / alpha` matrix-free.

Qiskit still prepares the variational system ansatz. QuBlock avoids constructing or statevector-simulating the enormous LCU circuit. This is not proof that Qiskit generally cannot solve this structured problem: a specialized simulator or a different hand-designed block encoding may use far fewer resources. The comparison is specifically against dense statevector simulation of the derived Pauli-LCU construction below, and this section does not construct its 262,144-term circuit.

In [ ]:
def diagonal_pauli_coefficients(diagonal: np.ndarray) -> np.ndarray:
    """Walsh-Hadamard transform giving coefficients of diagonal Pauli-Z strings."""
    coefficients = np.asarray(diagonal, dtype=float).copy()
    width = 1
    while width < coefficients.size:
        blocks = coefficients.reshape(-1, 2 * width)
        left = blocks[:, :width].copy()
        right = blocks[:, width:].copy()
        blocks[:, :width] = left + right
        blocks[:, width:] = left - right
        width *= 2
    return coefficients / coefficients.size


class GenericDiagonalOperator:
    def __init__(self, n_qubits: int, strength: float = 0.8, perturbation: float = 1e-4):
        self.n_qubits = int(n_qubits)
        self.strength = float(strength)
        self.dimension = 1 << self.n_qubits
        indices = np.arange(self.dimension, dtype=np.uint32)
        hamming_weight = np.fromiter(
            (int(index).bit_count() for index in indices),
            dtype=np.float64,
            count=self.dimension,
        )
        rng = np.random.default_rng(7)
        noise = perturbation * rng.uniform(-1.0, 1.0, self.dimension)
        self.diagonal = 1.0 + self.strength * hamming_weight / self.n_qubits + noise
        self.shape = (self.dimension, self.dimension)
        self.dtype = np.dtype(np.complex128)

    def apply(self, vector: np.ndarray) -> np.ndarray:
        return self.diagonal * vector

    def apply_into(self, vector: np.ndarray, out: np.ndarray) -> np.ndarray:
        np.multiply(self.diagonal, vector, out=out)
        return out

    def apply_adjoint(self, vector: np.ndarray) -> np.ndarray:
        return self.apply(vector)

    def apply_adjoint_into(self, vector: np.ndarray, out: np.ndarray) -> np.ndarray:
        return self.apply_into(vector, out)

    def norm_bound(self) -> float:
        return float(np.max(np.abs(self.diagonal)))


large_qubits = 18
large_operator = GenericDiagonalOperator(large_qubits)
large_coefficients = diagonal_pauli_coefficients(large_operator.diagonal)
large_term_count = int(np.count_nonzero(np.abs(large_coefficients) > 1e-12))
large_alpha = float(np.sum(np.abs(large_coefficients)))
large_index_ancillas = int(np.ceil(np.log2(large_term_count)))
large_phase_ancillas = int(np.any(large_coefficients < -1e-12))
large_ancillas = large_index_ancillas + large_phase_ancillas
large_block_encoding = BlockEncoding(
    op=large_operator,
    alpha=large_alpha,
    resources=ResourceEstimate(
        # Derived for prep_select from term count and coefficient phases.
        ancilla_qubits_clean=large_ancillas,
        postselections=1,
    ),
)

system_bytes = (1 << large_qubits) * np.dtype(np.complex128).itemsize
full_bytes = (1 << (large_qubits + large_ancillas)) * np.dtype(np.complex128).itemsize
print(f"system-only state: {system_bytes / 2**20:.1f} MiB")
print("nonzero Pauli-Z terms =", large_term_count)
print("derived prep_select ancillas =", large_ancillas)
print(f"full {large_qubits + large_ancillas}-qubit state: {full_bytes / 2**40:.1f} TiB")
print("constraint =", large_block_encoding.validate_quantum_constraints(require_verified=True).reason)

In [ ]:
def large_ansatz_state(theta: float) -> np.ndarray:
    circuit = QuantumCircuit(large_qubits)
    for qubit in range(large_qubits):
        circuit.ry(theta, qubit)
    return Statevector.from_instruction(circuit).data


large_b = np.full(1 << large_qubits, 1.0 / np.sqrt(1 << large_qubits), dtype=complex)


def large_vqls_cost(theta: float) -> float:
    state = StateVector(large_ansatz_state(theta))
    projected, _ = SemanticExecutor().run(
        Program([ApplyBlockEncodingQuantumStep(large_block_encoding, postselect=False)]),
        state,
    )
    y = projected.data
    return float(1.0 - abs(np.vdot(large_b, y)) ** 2 / np.vdot(y, y).real)


large_result = minimize_scalar(
    large_vqls_cost,
    bounds=(0.0, np.pi),
    method="bounded",
    options={"xatol": 1e-8},
)
large_x = large_ansatz_state(float(large_result.x))
large_exact = large_b / large_operator.diagonal
large_exact = large_exact / np.linalg.norm(large_exact)
large_fidelity = abs(np.vdot(large_exact, large_x)) ** 2

_, large_report = SemanticExecutor().run(
    Program([ApplyBlockEncodingQuantumStep(large_block_encoding, postselect=True)]),
    StateVector(large_x),
)

print("optimizer success =", large_result.success)
print("theta* =", large_result.x)
print("final large-system cost =", large_result.fun)
print("fidelity to exact normalized solution =", large_fidelity)
print("QuBlock state bytes =", large_x.nbytes)
print("tracked ancilla peak =", large_report.ancilla_clean_peak)
print("success probability =", large_report.cumulative_success_prob)
print("expected trials =", large_report.expected_trials)